# Washington State GHG Emissions: Who Bears the Burden?

**AD450 Final Project — Daniel Rice, Daniel Merced, Phakphoom, Veer**

## Thesis
The burden of industrial greenhouse gas emissions falls unevenly across Washington State. Rural, lower-income counties bear a disproportionate per-capita emissions load compared to wealthier, more populated urban counties.

## Datasets
1. **WA GHG Reporting Program** — Facility-level emissions reported to the state (2012–2023). Source: [data.wa.gov](https://data.wa.gov/)
2. **US Census County Population Estimates** — Annual county population (2012–2023). Source: [census.gov](https://www.census.gov/programs-surveys/popest/data/data-sets.html)
3. **SAIPE Income & Poverty Estimates** — Median household income and poverty rate by county (2024 snapshot). Source: [census.gov SAIPE](https://www.census.gov/programs-surveys/saipe.html)

## Guiding Questions
1. Which counties emit the most greenhouse gases *per person*, and how does that differ from total emissions?
2. How have emissions changed over time across different sectors?
3. Is there a relationship between a county's median income and its per-capita emissions burden?
4. Which industrial sectors dominate emissions in the counties that bear the heaviest per-capita burden?

---
## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
!pip install xlrd

In [ ]:
# Read the GHG raw data
ghg_url = "https://data.wa.gov/api/views/idhm-59de/rows.csv?accessType=DOWNLOAD"
ghg_path = "./data/raw/GHG_Reporting_Program_Publication.csv"

if os.path.exists(ghg_path):
    ghg_raw = pd.read_csv(ghg_path)
else:
    ghg_raw = pd.read_csv(ghg_url)
    ghg_raw.to_csv(ghg_path, index=False)

ghg_raw.head()


In [ ]:
# Census Data Sets

# Data for 2020-2024
raw_census_2020_to_2024 = pd.read_excel("data/raw/co-est2024-pop-53.xlsx", skiprows=3, header=0)

# Data for 2010-2019
raw_census_2010_to_2019 = pd.read_excel("data/raw/co-est2020int-pop-53.xlsx", skiprows=3, header=0)

In [ ]:
# Income Data Sets

raw_income = pd.read_excel("data/raw/est24all.xls", skiprows=3, header=0)

---
## 2. Exploratory Data Analysis (EDA)

Before cleaning or joining anything, we explore the raw GHG dataset to understand its shape, contents, and quirks.

### 2.1 Summarize the data (Daniel Rice)
Use `.info()`, `.shape`, and `.head()` to understand the structure of the GHG dataset — column names, dtypes, non-null counts, and a preview of actual values.

In [ ]:
# TODO (Daniel Rice): Summarize ghg_raw — .shape, .info(), .head()


### 2.2 Basic statistics on numeric columns (Daniel Merced)
Use `.describe()` on the numeric emissions columns. What is the range of reported emissions? What does the distribution look like (mean vs median)? Are there zeros?

In [ ]:
# TODO (Daniel Merced): .describe() on numeric columns, especially emissions columns
# Note any large differences between mean and median (indicates skew/outliers)


### 2.3 Value counts of categorical columns (Phakphoom)
Use `.value_counts()` on Sector, County, and City to understand the categorical distribution. How many unique sectors? Which counties appear most often? Are there any unexpected values (e.g., out-of-state counties)?

In [ ]:
# TODO (Phakphoom): value_counts() for Sector, County, and City
# Flag any counties that are NOT in Washington State
print(raw_census_2010_to_2019[~raw_census_2010_to_2019['Unnamed: 0'].str.contains('Washington')].shape[0])
print(raw_census_2020_to_2024[~raw_census_2020_to_2024['Unnamed: 0'].str.contains('Washington')].shape[0])
print(raw_income[~raw_income['Postal Code'].str.contains('WA')].shape[0])

### 2.4 Histograms of numeric columns (Veer)
Plot histograms of `Reported Emissions (MTCO2e)` and at least one gas breakdown column (e.g., Carbon Dioxide, Methane). Are emissions normally distributed or heavily skewed?

In [ ]:
# TODO (Veer): Histograms of Reported Emissions and at least one gas breakdown column
# Consider using log scale if the distribution is extremely skewed


---
## 3. Data Cleaning & Transformation

The raw data has null values, dtype issues, out-of-state rows, and columns we need to derive. We clean all three datasets in this section.

### 3.1 Correct data dtype issues (Daniel Rice)
- Ensure `Year` is int (not float).
- Parse the `Location` column (lat, lon string) into separate `Latitude` and `Longitude` float columns.
- Fix `Primary NAICS Code` to a clean numeric or string type.
- Also clean the census population dataframes: rename the first column to `County`, strip the leading dot and trailing `, Washington` from county names, drop the state total row, and ensure year columns are int.

In [ ]:
# TODO (Daniel Rice): Fix dtypes in ghg_raw
# Also clean raw_census_2010_to_2019 and raw_census_2020_to_2024:
#   - Rename first col to 'County'
#   - Strip leading '.' and trailing ' County, Washington' from county names
#   - Drop the state total row (first data row = 'Washington')
#   - Keep only year columns we need (2012-2019 from the first file, 2020-2023 from the second)


### 3.2 Fill NaN values (Daniel Merced)
The gas breakdown columns (Carbon Dioxide, Methane, Nitrous Oxide, HFCs, PFCs, SF6, Fluorinated-Other) have 67 null rows each. Decide on a fill strategy:
- If the row has a `Reported Emissions` value but missing breakdowns, fill gas columns with 0 (they weren't reported, not necessarily absent).
- The `Jurisdiction` column has ~232 nulls — fill with `'Unknown'` or investigate.
- Document your reasoning.

In [ ]:
# TODO (Daniel Merced): Fill NaN values in emissions breakdown columns and Jurisdiction
# Print null counts before and after to show the work


### 3.3 Remove inaccurate / out-of-scope data (Phakphoom)
The County value_counts revealed non-WA counties (e.g., `Greater Vancouver`, `Umatilla`, `Sonoma`, `Los Angeles`). These are facilities that report to WA but are physically located elsewhere.
- Identify and remove rows where County is not a valid WA county.
- Also remove the `Supplier` sector rows — these represent fuel suppliers, not physical emitters in a location, and will distort per-capita analysis.
- Document how many rows are removed and why.

In [ ]:
# TODO (Phakphoom): Remove out-of-state counties and Supplier sector rows
# Print row count before and after, and list the removed counties


### 3.4 Add derivative columns (Veer)
Create new columns that will power our analysis:
- `Non_CO2_Emissions`: sum of Methane + Nitrous Oxide + HFCs + PFCs + SF6 + Fluorinated-Other (shows how much of each facility's footprint is non-CO2 gases)
- Clean the income data: filter `raw_income` to WA counties only (`State FIPS Code == 53`, `County FIPS Code != 0`), strip `' County'` from the `Name` column, keep only `Name`, `Poverty Percent, All Ages`, and `Median Household Income`.

In [ ]:
# TODO (Veer): Add Non_CO2_Emissions column to ghg_raw
# Also clean raw_income into a tidy income_df with columns: County, Poverty_Pct, Median_Income


---
## 4. Data Joining

We now combine our three datasets to enable per-capita and income-based analysis.

### 4.1 Concatenate census population dataframes (Daniel Rice)
Melt each cleaned census dataframe from wide to long format (columns: `County`, `Year`, `Population`), then concatenate them into a single `population_df` covering 2012–2023.
- Watch for the 2020 overlap between the two files — pick one.
- Verify shape: should be 39 counties × 12 years = 468 rows.

In [ ]:
# TODO (Daniel Rice): pd.melt() each census df, then pd.concat() into population_df
# Confirm .shape and print .head()


### 4.2 Merge GHG data with population on County + Year (Daniel Merced)
Use `pd.merge()` to join the cleaned GHG data with `population_df` on `County` and `Year`. This gives every emissions row a population context.
- Use a left merge to keep all GHG rows.
- Check how many rows have null Population after the merge (counties that don't match).

In [ ]:
# TODO (Daniel Merced): pd.merge() ghg data with population_df on County + Year
# Check for null Population values after merge


### 4.3 Merge with income data on County (Phakphoom)
Use `pd.merge()` to join the income/poverty data onto the GHG+population dataframe on `County`.
- This is a many-to-one join (many GHG rows per county, one income row per county).
- Verify the join worked by spot-checking a known county like King or Lewis.

In [ ]:
# TODO (Phakphoom): pd.merge() to add Median_Income and Poverty_Pct columns
# Spot-check a couple of counties


### 4.4 Join aggregated dataframes on index (Veer)
To demonstrate index-based joining:
- Create a df of total emissions per county (all years summed), indexed by County.
- Create a df of average population per county (across all years), indexed by County.
- Use `.join()` to combine them on the County index.
- Add a `Per_Capita_Emissions` column (total emissions / avg population).

In [ ]:
# TODO (Veer): Create two indexed dfs and .join() them, then compute Per_Capita_Emissions
# This df will be key for the final visualizations


---
## 5. Aggregation & Grouping Operations

### 5.1 Aggregation on all data (Daniel Rice)
Compute total reported emissions across all of WA for each year (2012–2023). Is the state's total emissions trending up or down?

In [ ]:
# TODO (Daniel Rice): Group by Year, sum Reported Emissions across all counties/facilities
# Display as a simple table


### 5.2 Aggregation on a groupby (Daniel Merced)
Group by `Sector` and `Year`, then sum emissions. This reveals which industries are growing or shrinking their footprint over time.

In [ ]:
# TODO (Daniel Merced): groupby Sector + Year, sum Reported Emissions
# Display the top 5 sectors' trends


### 5.3 Pivot table — emissions by county and year (Phakphoom)
Create a pivot table with counties as rows, years as columns, and total reported emissions as values. This gives a complete emissions matrix for WA.

In [ ]:
# TODO (Phakphoom): pd.pivot_table() — index=County, columns=Year, values=Reported Emissions, aggfunc=sum
# Show the top 10 counties


### 5.4 Cross-tabulation — sector presence by county (Veer)
Create a cross-tabulation showing how many reporting facilities exist for each sector in each county. Which counties have the most diverse industrial base? Which are dominated by a single sector?

In [ ]:
# TODO (Veer): pd.crosstab() — County vs Sector, showing facility counts
# Filter to top 10 counties for readability


---
## 6. Data Visualization & Analysis

We now answer our guiding questions with polished visualizations.

### Q1: Which counties emit the most per person, and how does that differ from total emissions? (Daniel Rice)

Create a side-by-side bar chart (or two ranked bar charts) comparing:
- Top 10 counties by **total** emissions
- Top 10 counties by **per-capita** emissions

The key finding should be visible: counties like King lead in total emissions but are low per-capita, while rural/industrial counties like Lewis or Cowlitz dominate per-capita.

In [ ]:
# TODO (Daniel Rice): Side-by-side comparison of total vs per-capita emissions rankings
# Use clear titles, axis labels, and formatting


*TODO (Daniel Rice): Write 2-3 sentences interpreting what this visualization reveals.*

### Q2: How have emissions changed over time by sector? (Daniel Merced)

Create a line chart showing total emissions per year for the top 5 sectors. Are any sectors declining? Did COVID (2020) have a visible impact? Annotate or highlight any notable trends.

In [ ]:
# TODO (Daniel Merced): Multi-line chart of emissions over time by sector
# Include legend, axis labels, title, and any annotations


*TODO (Daniel Merced): Write 2-3 sentences interpreting the trends visible in this chart.*

### Q3: Is there a relationship between county income and per-capita emissions? (Phakphoom)

Create a scatter plot with:
- X-axis: Median Household Income
- Y-axis: Per-capita emissions
- Optional: size or color by population

Label notable outlier counties. Does lower income correlate with higher emissions burden?

In [ ]:
# TODO (Phakphoom): Scatter plot — Median Income vs Per-Capita Emissions
# Label key outlier counties, add title and axis labels


*TODO (Phakphoom): Write 2-3 sentences interpreting the income-emissions relationship. Does this support or challenge the environmental justice thesis?*

### Q4: Which sectors dominate emissions in the highest per-capita counties? (Veer)

Take the top 5 counties by per-capita emissions. Create a stacked bar chart showing the sector breakdown of emissions in each county. Are these counties dominated by one industry, or is the burden spread across sectors?

In [ ]:
# TODO (Veer): Stacked bar chart — sector breakdown for top 5 per-capita counties
# Clear legend, title, and axis labels


*TODO (Veer): Write 2-3 sentences interpreting which industries drive the burden in high per-capita counties.*

---
## 7. Conclusion

*TODO (All): Summarize the key findings:*
- *How does the per-capita picture differ from the total emissions picture?*
- *What role does income play in who bears the emissions burden?*
- *Which sectors are the biggest contributors in the hardest-hit counties?*
- *What are the limitations of this analysis?*